# Lab 05 solution · Restartable optical batch

Completed schema, repairs, investigations and offline extensions for the supplied
Sentinel-2 Collection 1 fixtures. The code cells retain the catalogue history and
save evidence before recovery. Source instructions are in `README.md` and
`SCHEMA_CONTRACT.md`. Start here from the `lab05_solution` directory.

The six requests cover three acquisition dates and two AOIs near Kraków.
Cloud suitability remains unassessed throughout this exercise.


## 1. Open the package and inspect the evidence

Run the notebook from the Lab05 directory. Keep the supplied Python files beside it.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, sqlite3, subprocess, sys
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import batch_plan, batch_repairs, processing_store

ROOT = Path.cwd().resolve()
assert (ROOT / 'lab05_cli.py').is_file(), 'Open this notebook from lab05_solution.'
EVIDENCE = ROOT / 'evidence'
EVIDENCE.mkdir(exist_ok=True)

def cli(*args, expected=(0,)):
    child = subprocess.run([sys.executable, str(ROOT / 'lab05_cli.py'), *map(str, args)], capture_output=True, text=True)
    try:
        response = json.loads(child.stdout) if child.stdout.strip() else None
    except json.JSONDecodeError:
        response = {'stdout': child.stdout}
    with (EVIDENCE / 'commands.jsonl').open('a') as stream:
        stream.write(json.dumps({'recorded_at': processing_store.now(), 'args': list(map(str, args)),
                                'exit_code': child.returncode, 'response': response,
                                'stderr': child.stderr}) + '\n')
    assert child.returncode in expected, (child.returncode, child.stdout[-2500:], child.stderr[-2500:])
    if child.returncode == 87:
        return {'exit_code': 87, 'note': 'The isolated coordinator stopped abruptly.'}
    return response

def check(group='all'):
    child = subprocess.run([sys.executable, str(ROOT / 'check_lab05.py'), group], capture_output=True, text=True)
    print(child.stdout)
    return child.returncode == 0

def query(manifest_path, sql, params=()):
    manifest = batch_plan.load(manifest_path)
    with sqlite3.connect(manifest['_database']) as connection:
        connection.row_factory = sqlite3.Row
        connection.execute('PRAGMA foreign_keys=ON')
        return [dict(row) for row in connection.execute(sql, params)]

SQL = []
statement = ''
for line in (ROOT / 'investigation_queries.sql').read_text().splitlines(True):
    statement += line
    if sqlite3.complete_statement(statement):
        SQL.append(statement)
        statement = ''
assert len(SQL) == 6

def investigate(manifest_path, number, **params):
    manifest = batch_plan.load(manifest_path)
    return query(manifest_path, SQL[number - 1], {'batch_id': manifest['batch_id'], **params})

LATEST = """SELECT m.member_id, o.observation_id, o.state, o.reason, o.observed_at, o.evidence_json
FROM batch_members m LEFT JOIN batch_observations o
ON o.batch_id=m.batch_id AND o.member_id=m.member_id
AND o.observation_id=(SELECT MAX(x.observation_id) FROM batch_observations x
                     WHERE x.batch_id=m.batch_id AND x.member_id=m.member_id)
WHERE m.batch_id=:batch_id ORDER BY m.member_id"""

def capture(name, manifest_path):
    path = EVIDENCE / (name + '.json')
    if path.exists():
        return json.loads(path.read_text())
    manifest = batch_plan.load(manifest_path)
    latest = query(manifest_path, LATEST, {'batch_id': manifest['batch_id']})
    summary = batch_repairs.summarize_batch([{**row, 'observation_id': row['observation_id'] if row['observation_id'] is not None else -1} for row in latest])
    value = {'captured_at': processing_store.now(), 'batch_id': manifest['batch_id'],
             'manifest': str(manifest_path), 'latest': latest, 'summary': summary,
             'attempts': query(manifest_path, 'SELECT * FROM processing_attempts ORDER BY started_at, attempt_id'),
             'no_attempts': investigate(manifest_path, 1), 'recovered_requests': investigate(manifest_path, 2),
             'unavailable_products': investigate(manifest_path, 3),
             'missing_roles': investigate(manifest_path, 5), 'running': investigate(manifest_path, 6)}
    path.write_text(json.dumps(value, indent=2))
    return value

def saved_report(manifest_path, name):
    directory = ROOT / 'reports' / name
    if not directory.exists():
        cli('report', '--manifest', manifest_path, '--output', directory)
    return json.loads((directory / 'report.json').read_text())

print('Python', sys.version.split()[0], 'Rasterio', rasterio.__version__)
fixture = json.loads((ROOT / 'fixtures/manifest.json').read_text())
print('Endpoint:', fixture['endpoint'], 'Collection:', fixture['collection'])
print('Requests:', len(fixture['members']))


In [ ]:
first_member = fixture['members'][0]
for role, entry in first_member['bands'].items():
    with rasterio.open(ROOT / 'fixtures' / entry['file']) as source:
        raw = source.read(1, masked=True)
        print(role, first_member['member_id'], source.shape, source.crs, source.scales, source.offsets, entry['sha256'])
        print('DN median:', float(np.ma.median(raw)))
print('Source transfer bytes:', sum(t['network_bytes'] for t in fixture['transfers']))

Name one property a checksum establishes and one scientific question it cannot answer. Explain why six requests here represent three dates rather than six independent acquisitions.

A matching SHA256 establishes that the measured bytes match the recorded
digest, assuming that reference digest is trusted. It cannot establish that the
scene is cloud free, correctly calibrated, or suitable for vegetation analysis.

There are three Sentinel-2 acquisitions, on 4, 5 and 10 June 2025. Each supplies
two AOIs, so six processing requests share three acquisition dates. The AOIs do
not become independent acquisitions because they have separate request IDs.


## 2. Design the new tables

Read `SCHEMA_CONTRACT.md`. Write all three tables in `student_batch_schema.sql`. The earlier acquisition and processing schemas are completed in `support/`.

Before writing SQL, sketch the links between a batch, its expected members, their requests, attempts and observations. Decide which records must exist when we know what we want but no calculation has started.

A member's planned request ID exists before its optional request foreign key can be filled. Explain why those are separate fields. Then run the schema check.

In [ ]:
assert check('schema')


`batches` stores the frozen scientific membership and budget. `batch_members`
stores all six expected requests before any processing starts. Each member can
then acquire several `batch_observations`, while one request can have several
historical attempts and producing outputs.

```text
batch -> expected member -> planned request identity
                       -> optional registered request -> attempts -> outputs
                       -> observations -> optional explanatory attempt
```

`planned_request_id` describes the work we expect. Its processing request row
need not exist yet. `request_id` is initially NULL and becomes a foreign key
when the coordinator registers the request; a CHECK prevents it from disagreeing
with the planned identity. Making the planned field a foreign key would prevent
recording work that has never started. The composite member key also anchors
observations, so unknown members cannot receive a state.


## 3. Repair the coordination logic

The functions in `batch_repairs.py` execute, but the supplied versions make incorrect decisions. Read their contracts and repair them.

`classify_member` needs to distinguish blocked inputs, unresolved execution, verified completion and a permitted new attempt. A finished result may remain usable after its inputs move. An unresolved active execution needs attention even when an older result exists. The attempt budget does not reset when the coordinator restarts.

`verify_result` must reject evidence for another request, mismatching bytes, or missing verification. The adapter does the actual catalogue queries, hashing and raster inspection.

`summarize_batch` must count expected members once each using their newest observations. Repeated observations and joined locations do not create more requested work. Keep members that have no observation yet.

Predict which check will expose each supplied defect before editing.

In [ ]:
for group in ('classify', 'verify', 'summary'):
    assert check(group)


The classify check exposes missing input being called failed, ignored
validation errors, blind retries after ordinary failures, and budget checks
overriding a verified result or an active attempt. I check active execution
first, then verified completion, validation, missing input, budget and the last
failure. Reconciled interruptions can use the remaining budget.

The verify check exposes acceptance based only on an artifact ID. The repair
requires the exact request, nonempty attempt/artifact IDs, positive matching
sizes and SHA256, and all three verification flags explicitly set to True.
The supplied adapter obtains this evidence from catalogue joins and actual
file and raster checks.

The summary check exposes counting joined rows and dropping unobserved members.
The repair chooses the largest observation ID for each member, counts it once,
and sorts each state's member list. `test_solution.py` also checks absent
verification fields, malformed evidence and stricter schema boundaries.


## 4. Register expected work and run the batch

Finish the schema and function repairs before creating processing evidence. The next cell imports the captured real inputs into a fresh catalogue and freezes membership. It does not make a processing attempt.

Rerunning the notebook keeps existing work. To begin a separate experiment, choose a new workspace name.

In [ ]:
assert check('all')
MANIFEST = ROOT / 'work/batch.json'
if not MANIFEST.exists():
    print(cli('setup', '--directory', ROOT / 'work'))
initial = capture('main_before', MANIFEST)
assert initial['summary']['expected'] == 6
assert len(initial['attempts']) == 0
assert len(initial['no_attempts']) == 6
initial['no_attempts']


Write a query that lists expected members with no application attempt. Run it now. It should include the whole batch. Keep the query, its parameters and your interpretation in `investigation_queries.sql`.

In [ ]:
# Q1 uses planned identity, so NULL registration cannot hide expected work.
print('Before the first run:', initial['no_attempts'])
print('Current catalogue:', investigate(MANIFEST, 1))


In [ ]:
database = batch_plan.load(MANIFEST)['_database']
before_plan = processing_store.file_hash(database)
plan = cli('plan', '--manifest', MANIFEST)
assert not plan['writes_database']
assert processing_store.file_hash(database) == before_plan
print([(m['member_id'], m['state'], m['reason']) for m in plan['members']])
result = cli('run', '--manifest', MANIFEST)
assert result['complete']
print('Complete:', result['complete'], 'New invocations:', len(result['invocations']))


Predict what a second invocation will change. Will it create another scientific product, another application attempt, or only new batch observations?

In [ ]:
attempts_before = query(MANIFEST, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
repeat = cli('run', '--manifest', MANIFEST)
attempts_after = query(MANIFEST, 'SELECT COUNT(*) AS n FROM processing_attempts')[0]['n']
print('New invocations:', len(repeat['invocations']))
assert repeat['complete'] and not repeat['invocations']
assert attempts_before == attempts_after

On the first run, all six requests produce an output and one succeeded
application attempt each. A second run verifies those files again and appends
batch observations. It launches no application attempts and publishes no new
scientific product. The persistent attempt count remains six. The planning
cell also hashes the database before and after `plan` to check that planning
does not write it.


## 5. Look at the results and their report

A contact sheet makes the six results easy to compare. The saved raster values are not clipped. The display uses a fixed interval; blue and magenta identify values outside it.

Export a report, then open its HTML file. Follow one count to its members and inspect a request, producing attempt and verification time. Scientific cloud suitability should remain unassessed.

In [ ]:
report = saved_report(MANIFEST, 'main')
print(report['summary'])
assert report['summary']['counts']['complete'] == 6
assert report['attempt_count'] == 6
from IPython.display import Image, display
display(Image(filename=str(ROOT / 'reports/main/preview.png')))
print('Open:', ROOT / 'reports/main/index.html')
main_after = capture('main_after', MANIFEST)
print('Generated:', report['generated_at'])
print('Observation times:', [m['observed_at'] for m in report['members']])
print('Verification times:', [m['evidence']['result']['checked_at'] for m in report['members']])


If the report says complete, what has it established? Give two reasons that still would not justify calling the image a clear sky vegetation observation. Identify the observation times separately from the report generation time.

Complete means that a succeeded producing attempt has an NDVI output whose
bytes, grid and request/attempt tags passed verification at the recorded time.
It establishes delivery for that frozen request.

The mask policy is `nodata_only`: cloud and shadow screening were not performed.
Finite values can also represent water, buildings or mixed pixels rather than
vegetation. Even an in-range NDVI and high finite coverage cannot establish a
clear sky vegetation observation. Calibration accuracy needs its own review.

Each member has `observed_at`, its output evidence has `checked_at`, and each
location check has another `checked_at`. The report's `generated_at` is the
export time. Opening or re-exporting the report does not refresh those checks.
